In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

urls = [
     "https://inside.fifa.com/fifa-world-ranking/men?dateId=id14541",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=id14576",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=id14597",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=id14702",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=id14800",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=id14870",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=FRS_Male_Football_20250910",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=FRS_Male_Football_20251015",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=FRS_Male_Football_20251119",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=FRS_Male_Football_20251219",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=FRS_Male_Football_20260119",
    "https://inside.fifa.com/fifa-world-ranking/men?dateId=id14541",
]

# Configura o navegador para resolução Desktop (Garante que a coluna de pontos apareça)
chrome_options = Options()
chrome_options.add_argument("--window-size=1920,1080")
# chrome_options.add_argument("--headless") # Descomente esta linha se quiser que o navegador rode em segundo plano (invisível)

driver = webdriver.Chrome(options=chrome_options)
all_data = []

for url in urls:
    date_id = url.split("dateId=")[-1]
    print(f"Raspando dados de: {date_id} ...")
    
    driver.get(url)
    
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr"))
        )
        time.sleep(2) 
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        table = soup.find("table")
        
        if table and table.find("tbody"):
            rows = table.find("tbody").find_all("tr")
            
            for row in rows:
                cols = row.find_all("td")
                
                if len(cols) >= 3:
                    # O 'separator=" "' evita que textos de spans diferentes fiquem grudados
                    textos = [col.get_text(separator=" ", strip=True) for col in cols]
                    
                    # 1. Corrigindo o Rank: Pega apenas o primeiro número antes do espaço
                    # Exemplo: "1 0" -> "1", "10 2" -> "10"
                    rank_raw = textos[0]
                    rank = rank_raw.split()[0] if rank_raw else ""
                    
                    # 2. Equipe
                    team = textos[1]
                    
                    # 3. Corrigindo os Pontos: Busca nas colunas o padrão de pontos (ex: 1855.20)
                    points = ""
                    for txt in textos[5:]:
                        # Procura números com duas casas decimais no formato FIFA
                        match = re.search(r'\d{1,4}\.\d{2}', txt)
                        if match:
                            points = match.group(0)
                            break
                    
                    # Se não achar o padrão exato, salva a 3ª coluna como segurança
                    if not points and len(textos) > 2:
                        points = textos[2]

                    all_data.append({
                        "DateID": date_id,
                        "Rank": rank,
                        "Team": team,
                        "Points": points,
                        "Raw_Data": " | ".join(textos) # Coluna de segurança para auditoria
                    })
        else:
            print(f"Tabela não encontrada em {url}")
            
    except Exception as e:
        print(f"Erro ao extrair {url}. Motivo: {e}")

driver.quit()

df = pd.DataFrame(all_data)
# Removendo a coluna de auditoria para limpar o CSV final
df = df.drop(columns=['Raw_Data']) 
df.to_csv("historico_ranking_fifa_corrigido.csv", index=False, encoding="utf-8")

print("\nConcluído! Dados limpos salvos em 'historico_ranking_fifa_corrigido.csv'.")

Raspando dados de: id14541 ...
Raspando dados de: id14576 ...
Raspando dados de: id14597 ...
Raspando dados de: id14702 ...
Raspando dados de: id14800 ...
Raspando dados de: id14870 ...
Raspando dados de: FRS_Male_Football_20250910 ...
Raspando dados de: FRS_Male_Football_20251015 ...
Raspando dados de: FRS_Male_Football_20251119 ...
Raspando dados de: FRS_Male_Football_20251219 ...
Raspando dados de: FRS_Male_Football_20260119 ...
Raspando dados de: id14541 ...

Concluído! Dados limpos salvos em 'historico_ranking_fifa_corrigido.csv'.


In [9]:
async def scrape_fifa_ranking_url(url: str) -> pd.DataFrame:
    date_id = get_date_id(url)
    json_payloads = []

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=True)
        page = await browser.new_page(
            viewport={"width": 1440, "height": 1200},
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
        )

        async def handle_response(response):
            content_type = response.headers.get("content-type", "")

            if "application/json" not in content_type:
                return

            try:
                payload = await response.json()
                json_payloads.append(
                    {
                        "url": response.url,
                        "payload": payload,
                    }
                )
            except Exception:
                return

        page.on("response", handle_response)

        await page.goto(url, wait_until="networkidle", timeout=90_000)
        await page.wait_for_timeout(3_000)

        try:
            show_full_button = page.get_by_text("Show full rankings", exact=False)

            if await show_full_button.count() > 0:
                await show_full_button.first.click(timeout=10_000)
                await page.wait_for_load_state("networkidle", timeout=60_000)
                await page.wait_for_timeout(2_000)
        except Exception:
            pass

        page_text = await page.locator("body").inner_text(timeout=30_000)
        html = await page.content()

        await browser.close()

    debug_json_path = debug_dir / f"{date_id}_json_payloads.json"
    debug_html_path = debug_dir / f"{date_id}.html"

    debug_json_path.write_text(
        json.dumps(json_payloads, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    debug_html_path.write_text(html, encoding="utf-8")

    rank_date = parse_rank_date_from_date_id(date_id)

    if rank_date is None:
        rank_date = parse_rank_date_from_page_text(page_text)

    if date_id == "LiveRanking" and rank_date is None:
        rank_date = pd.Timestamp.today().normalize()

    ranking_rows = []

    for payload_data in json_payloads:
        ranking_rows = extract_ranking_rows_from_json(payload_data["payload"])

        if len(ranking_rows) >= 20:
            break

    if len(ranking_rows) < 20:
        ranking_rows = extract_ranking_rows_from_html(html)

    if len(ranking_rows) < 20:
        raise ValueError(
            f"Não consegui extrair ranking suficiente para {date_id}. "
            f"Arquivos de debug salvos em: {debug_dir}"
        )

    ranking_data = pd.DataFrame(ranking_rows)
    ranking_data["rank_date"] = rank_date
    ranking_data["date_id"] = date_id
    ranking_data["source_url"] = url
    ranking_data["is_live"] = date_id == "LiveRanking"

    ranking_data = ranking_data[
        [
            "rank_date",
            "team",
            "rank",
            "total_points",
            "date_id",
            "is_live",
            "source_url",
        ]
    ].copy()

    ranking_data = ranking_data.drop_duplicates(
        subset=["rank_date", "team"],
        keep="last",
    )

    return ranking_data.sort_values("rank").reset_index(drop=True)